<a href="https://colab.research.google.com/github/SamRB-dev/DeepLearningLab/blob/main/lab/lab_02_UPPAAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPPAAL and Timed Automata: Introductory Lab
**Mode:** Theory → example → small modeling exercise in UPPAAL  

This notebook is a guided lab. It is **not** meant to execute UPPAAL models inside Jupyter.  
Use Jupyter for reading, notes, and answering questions. Use the **UPPAAL GUI** to create, simulate, and verify the models.

## References

This lab is based primarily on:

- G. Behrmann, A. David, K. G. Larsen, **A Tutorial on UPPAAL**, available from the UPPAAL website.
- UPPAAL Documentation: <https://docs.uppaal.org/>

Key ideas used here:
- UPPAAL verifies real-time systems modeled as networks of timed automata.
- Timed automata are finite-state machines extended with clocks.
- UPPAAL models include clocks, bounded integer variables, arrays, templates, and synchronization channels.
- UPPAAL queries are based on a simplified subset of CTL and include reachability, safety, liveness, and deadlock checks.

---
# 1. Timed Automata: Basic Idea

## Theory

A **timed automaton** is a finite-state machine extended with **clock variables**.

A model consists of:

| Concept | Meaning |
|---|---|
| Location | A control state, e.g. `Idle`, `Waiting`, `Working` |
| Initial location | The starting location |
| Clock | A real-valued variable that increases with time |
| Edge | A transition between locations |
| Guard | Condition that must be true for an edge to be taken |
| Reset | Assignment that sets a clock to zero |
| Invariant | Condition that restricts how long the automaton may remain in a location |

UPPAAL models time as **dense time**: clocks progress continuously and synchronously.

## Example: Simple Timer

A process starts in `Idle`. It may move to `Waiting` and reset a clock `x`.  
It may leave `Waiting` only after at least 5 time units.

UPPAAL idea:

```text
clock x;

Idle -> Waiting:
    update x = 0

Waiting -> Done:
    guard x >= 5
```

## Exercise 1: Door Timer

Model a door controller in UPPAAL:

- Locations: `Closed`, `Open`, `Alarm`
- The door may open from `Closed`
- When the door opens, reset clock `x`
- If the door remains open for more than 10 time units, go to `Alarm`
- The door may close again before 10 time units

Answer:
1. Which locations do you need? **Answer:** closed, open, alarm.
2. Which clock do you need? **Answer:** x/clk
3. Which guards and resets do you use? **Answer:** clk = 0
4. Which invariant could you use if the system must react exactly at 10? **Answer:** in Open, invariant, clk <= 10

---
# 2. Guards, Resets, and Invariants

## Theory

A **guard** enables an edge only if a condition is true.

Examples:
```c
x >= 5
x <= 10
count > 0
```

A **reset** sets a clock to zero:
```c
x = 0
```

An **invariant** restricts staying in a location:
```c
x <= 10
```

Important distinction:

- A guard controls **when an edge may be taken**.
- An invariant controls **how long a location may be occupied**.

## Example: Processing Task

A task must process for at least 3 and at most 7 time units.

Locations:
```text
Idle -> Processing -> Done
```

Declarations:
```c
clock x;
```

Transitions:
```text
Idle -> Processing:
    update x = 0

Processing -> Done:
    guard x >= 3
```

Location invariant:
```text
Processing invariant: x <= 7
```

## Exercise 2: Sensor Sampling

Model a sensor task:

- It starts in `Idle`
- It moves to `Sampling` and resets clock `x`
- Sampling must take at least 2 and at most 5 time units
- Afterwards the task returns to `Idle`

Answer:
1. Give the locations.
2. Give the clock declaration.
3. Give the guard and invariant.
4. Explain what happens if the invariant is missing.

---
# 3. Networks of Timed Automata and Synchronization

## Theory

UPPAAL systems are usually **networks of timed automata**.  
Each template describes one kind of process. Several process instances run in parallel.

UPPAAL supports binary synchronization channels:

```c
chan start;
```

One edge uses:
```text
start!
```

Another edge uses:
```text
start?
```

Both edges synchronize and happen together.

## Example: Controller Starts a Worker

Global declaration:
```c
chan start;
```

Controller:
```text
Idle -> Sent:
    sync start!
```

Worker:
```text
Idle -> Working:
    sync start?
```

The worker can enter `Working` only when the controller sends `start!`.

## Exercise 3: Request and Release

Model a worker and controller:

- The worker requests permission using `request!`
- The controller receives it using `request?`
- The controller later releases the worker using `release!`
- The worker receives it using `release?`
- The worker may only enter `Working` after receiving `release?`

Answer:
1. Which channels do you declare?
2. Which locations does the worker need?
3. Which locations does the controller need?
4. Which synchronizations do you use?

---
# 4. Variables, Bounded Integers, Arrays, and Functions

## Theory

UPPAAL allows discrete variables similar to programming languages.

Examples:
```c
int[0,5] count = 0;
const int N = 2;
int[0,N-1] id;
```

Bounded integer variables are important because verification explores state spaces.  
Large or unbounded variables can make verification inefficient or impossible.

Arrays are also possible:

```c
const int QSIZE = 4;
int[0,1] queue[QSIZE];
int[0,QSIZE] head = 0;
int[0,QSIZE] tail = 0;
int[0,QSIZE] count = 0;
```

Functions can help modularize model logic:

```c
void enqueue(int id) {
    queue[tail] = id;
    tail = (tail + 1) % QSIZE;
    count++;
}
```

## Example: Counter

A process can increment `count` while `count < 3`.

```text
guard count < 3
update count++
```

## Exercise 4: Bounded Buffer Counter

Model a bounded buffer only by its size:

- Capacity is 3
- Producer increments `count`
- Consumer decrements `count`
- The producer must not produce if full
- The consumer must not consume if empty

Answer:
1. Give the global declaration of `count`.
2. Give the producer guard/update.
3. Give the consumer guard/update.
4. Give two verification queries for overflow and underflow.

---
# 5. Urgent and Committed Locations

## Theory

UPPAAL supports three location types:

| Type | Meaning |
|---|---|
| Normal | Time may pass if invariant allows it |
| Urgent | Time may not pass in the location |
| Committed | Time may not pass, and the next transition must involve a committed location |

Use urgent or committed locations carefully. They are useful for modeling atomic or immediate behavior, but overuse can make models unrealistic or difficult to understand.

## Example

A controller receives a request and must immediately dispatch it without allowing time to pass between receiving and selecting.

Possible model:
```text
Idle -> Select   (Select is committed)
Select -> Release
```

## Exercise 5: Immediate Dispatch

Extend the request/release model:

- After the controller receives a request, it enters `Select`
- `Select` should be committed
- The controller immediately sends `release!`

Answer:
1. Why could `Select` be committed?
2. What behavior is excluded by making it committed?
3. What is a risk of using committed locations too often?

---
# 6. UPPAAL Query Language: Timed CTL Fragment

## Theory

UPPAAL queries combine **state formulae** and **path formulae**.

A **state formula** describes one state:
```text
count == 0
Worker0.Working
x <= 5
```

A **path formula** quantifies over possible executions.

Common forms:

| Query | Meaning |
|---|---|
| `E<> φ` | There exists a path where eventually φ holds |
| `A[] φ` | For all paths, φ always holds |
| `E[] φ` | There exists a path where φ always holds |
| `A<> φ` | For all paths, eventually φ holds |
| `φ --> ψ` | Whenever φ holds, ψ will eventually hold |

UPPAAL also provides the special state formula:
```text
deadlock
```

Typical deadlock freedom query:
```text
A[] not deadlock
```

## Example Queries

Reachability:
```text
E<> Worker0.Working
```

Safety:
```text
A[] count <= QSIZE
```

Deadlock freedom:
```text
A[] not deadlock
```

Response / leads-to:
```text
Worker0.Waiting --> Worker0.Working
```

## Exercise 6: Query Interpretation

Explain the meaning of these queries:

1. `E<> Alarm`
2. `A[] count <= 4`
3. `A[] not deadlock`
4. `Worker0.Waiting --> Worker0.Working`

Then formulate queries for:
5. "Worker 1 can reach Working."
6. "The buffer is never negative."
7. "Whenever a request is stored, eventually it can be released."

---
# 7. Integrated Modeling Exercise: Two Workers and FIFO Controller

## Theory

This exercise combines:

- networks of timed automata
- synchronization
- bounded integers
- arrays
- FIFO queue logic
- reachability, safety, deadlock, and response queries

## Problem

There are two workers and one controller.

Each worker:

1. requests permission to work,
2. waits for a task-specific release,
3. starts working only after release,
4. works for a bounded time,
5. returns to idle.

The controller:

1. receives worker requests,
2. stores worker IDs in a bounded FIFO queue,
3. dispatches workers in FIFO order,
4. sends a task-specific release signal.

## Suggested Declarations

```c
const int N = 2;
const int QSIZE = 4;

typedef int[0,N-1] task_t;
typedef int[0,QSIZE] qindex_t;

chan request[N];
chan release[N];

task_t queue[QSIZE];

qindex_t head = 0;
qindex_t tail = 0;
qindex_t count = 0;

task_t selected = 0;
```

## Suggested Queue Functions

```c
void enqueue(task_t id) {
    queue[tail] = id;
    tail = (tail + 1) % QSIZE;
    count++;
}

task_t dequeue() {
    task_t id;
    id = queue[head];
    head = (head + 1) % QSIZE;
    count--;
    return id;
}
```

## Exercise 7A: Worker Template

Create a parameterized worker template:

```c
Worker(const task_t id)
```

Required locations:

- `Idle`
- `Waiting`
- `Working`

Required behavior:

- `Idle -> Waiting` with `request[id]!`
- `Waiting -> Working` with `release[id]?`
- `Working -> Idle` after some bounded work time

Add a local clock for work timing.

## Exercise 7B: Controller Template

Create a controller template.

Required behavior:

- receive `request[0]?` and enqueue worker 0
- receive `request[1]?` and enqueue worker 1
- only enqueue if `count < QSIZE`
- dispatch only if `count > 0`
- dequeue the next worker into `selected`
- send `release[selected]!`

## Exercise 7C: Verification

Verify:

```text
A[] not deadlock
A[] count <= QSIZE
A[] count >= 0
E<> Worker0.Working
E<> Worker1.Working
Worker0.Waiting --> Worker0.Working
Worker1.Waiting --> Worker1.Working
```

Answer:
1. Which queries are safety properties?
2. Which queries are reachability properties?
3. Which queries are liveness/response properties?
4. Does your model prove fairness? Explain carefully.

---
# 8. Modeling Checklist

Before submitting an UPPAAL model, check:

- [ ] Are all clocks declared?
- [ ] Are clocks reset where timing intervals start?
- [ ] Are guards used for lower timing bounds?
- [ ] Are invariants used for upper timing bounds?
- [ ] Are variables bounded?
- [ ] Are arrays protected against invalid access?
- [ ] Are synchronization channels paired correctly (`!` with `?`)?
- [ ] Are urgent/committed locations used only when justified?
- [ ] Have you checked `A[] not deadlock`?
- [ ] Have you separated reachability, safety, and liveness properties?

---
# Student Notes

Use the following cells for your own notes. They are not executed as UPPAAL code.

In [ ]:
# Paste selected UPPAAL declarations or queries here as notes.
